In [196]:
import csv
import math
import nltk
nltk.download('semcor')
from nltk.corpus import wordnet as wn
import numpy as np
import re
from nltk.corpus import semcor
import random
import google.generativeai as genai
# type your api key 
from dotenv import load_dotenv
import os
import time

semcor_sentences = list(semcor.sents())
semcor_tagged_sentences = list(semcor.tagged_sents(tag="sem"))

genai.configure(api_key=os.environ['GEMINI_API_KEY'])

[nltk_data] Downloading package semcor to
[nltk_data]     C:\Users\kurai\AppData\Roaming\nltk_data...
[nltk_data]   Package semcor is already up-to-date!


In [197]:
# Set up the model
generation_config = {
  "temperature": 0.7,
  "top_p": 0.9,
  "top_k": 1,
  "max_output_tokens": 2048,
}

safety_settings = [
  {
    "category": "HARM_CATEGORY_HARASSMENT",
    "threshold": "BLOCK_MEDIUM_AND_ABOVE"
  },
  {
    "category": "HARM_CATEGORY_HATE_SPEECH",
    "threshold": "BLOCK_MEDIUM_AND_ABOVE"
  },
  {
    "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
    "threshold": "BLOCK_MEDIUM_AND_ABOVE"
  },
  {
    "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
    "threshold": "BLOCK_MEDIUM_AND_ABOVE"
  },
]

model = genai.GenerativeModel(model_name="gemini-1.0-pro",
                              generation_config=generation_config,
                              safety_settings=safety_settings)



In [259]:
def get_generated_sentences(word, synset, description):
    try:
        prompt_parts = [
            "Generate multiple sentences that use words from the synset related to {word} as {synset}.",
            "Ensure each sentence clearly reflects the meaning of the word {word} with sense {synset}.",
            "Use the following synset description as context: {description}.",
            "Provide at least 5 distinct sentences."
        ]

        formatted_prompt = [part.format(word=word, synset=synset, description=description) for part in prompt_parts]

        response = model.generate_content(formatted_prompt)

        cleaned_sentences = re.findall(r'^\d+\.\s*(.+)', response.text, re.MULTILINE)

        return cleaned_sentences
    except Exception as e:
        #print(f"An error occurred: {e}")
        return []


In [298]:
def remove_non_word_chars(sentence):
    return re.sub(r'\W+', ' ', sentence).strip()

def parse_for_signature(sentence):
    return set(remove_non_word_chars(sentence).lower().split(" "))

def ComputeOverlap(signature,context):
    inter = signature.intersection(context)
    return len(inter)

stop_words = []
with open('stop_words_FULL.txt') as f:
    stop_words = f.read().splitlines()

def SimplifiedLesk(word,sentence,gemini = False):
    #print("Simplified Lesk", word, sentence)
    best_sense = None
    max_overlap = 0
    context = set(parse_for_signature(sentence)).difference(stop_words)
    for sense in wn.synsets(word):
        if (best_sense is None):
            best_sense = sense
        signature = set()
        for example in sense.examples():
            signature.update(parse_for_signature(example))

        if gemini:
            for generated_sentence in get_generated_sentences(word, sense, sense.definition()):
                signature.update(parse_for_signature(generated_sentence))
            time.sleep(2)

        
        
        signature.update(parse_for_signature(sense.definition()))

        signature = signature.difference(stop_words)

        overlap = ComputeOverlap(signature,context)
        if overlap > max_overlap:
            max_overlap = overlap
            best_sense = sense
    return best_sense 
        


""" best = SimplifiedLesk("bank", "the bank can guarantee deposits will eventually cover future tuition costs because it invests in adjustable-rate mortgage securities")

print(best, best.definition()) """

' best = SimplifiedLesk("bank", "the bank can guarantee deposits will eventually cover future tuition costs because it invests in adjustable-rate mortgage securities")\n\nprint(best, best.definition()) '

In [293]:
def get_sentence_from_semcor(sentence_num):
   sentence_index = sentence_num
   sentence = " ".join(semcor_sentences[sentence_index])
   tags = semcor_tagged_sentences[sentence_index]
   word = None
   attempts = 0
   while(word == None):
        if attempts > 5:
            attempts = 0
            sentence_index += 1
            sentence = " ".join(semcor_sentences[sentence_index])
            tags = semcor_tagged_sentences[sentence_index]
        
        i = random.randint(0, len(tags)-1)
        if tags[i][0] not in stop_words and isinstance(tags[i], nltk.Tree) and isinstance(tags[i][0], str) and isinstance(tags[i].label(), nltk.corpus.reader.wordnet.Lemma):
            word = tags[i][0]
            target = tags[i].label().synset()
        attempts += 1
   return sentence, word, target

def get_random_semcor_sentences(n):    
    max_sentence = len(semcor_sentences)-1
    list_of_random_indexes = []
    while len(list_of_random_indexes)<n:
        test_index = random.randint(0, max_sentence)
        if test_index not in list_of_random_indexes:
            list_of_random_indexes.append(test_index)

    corpus_sentences = [get_sentence_from_semcor(i) for i in list_of_random_indexes]

    return corpus_sentences




In [294]:
def semcor_test(sentences, n_sentences, gemini):
    
    disambiguated_sentences = [SimplifiedLesk(sentence[1], sentence[0], gemini) for sentence in sentences]
    target_sentences = [sentence[2] for sentence in sentences]

    counter = 0
    for sentence, disambiguated, target in zip(sentences, disambiguated_sentences, target_sentences):
        if disambiguated and target and disambiguated == target:
            counter += 1
    
    return counter / n_sentences

In [295]:
def multiple_semcor_test(n_trials, n_sentences, sentences, gemini):    
    print("Running", n_trials)
    counter = 0;
    for i in range(n_trials):
        print("Starting test number", i + 1)
        test = semcor_test(sentences[i],n_sentences, gemini)
        counter += test
        print("Accuracy test", i + 1, ":", test)
    return counter / n_trials



In [302]:
n_sentences = 10
n_trials=5
random_sentences = []
for i in range(n_trials):
    random_sentences.append(get_random_semcor_sentences(n_sentences))


In [311]:
result_no_gemini = multiple_semcor_test(n_trials=n_trials, n_sentences=n_sentences, sentences=random_sentences, gemini=False)

print("Accuracy:", result_no_gemini)

Running 5
Starting test number 1
Accuracy test 1 : 0.5
Starting test number 2
Accuracy test 2 : 0.7
Starting test number 3
Accuracy test 3 : 0.4
Starting test number 4
Accuracy test 4 : 0.4
Starting test number 5
Accuracy test 5 : 0.6
Accuracy: 0.52


In [313]:

result_with_gemini = multiple_semcor_test(n_trials=n_trials, n_sentences=n_sentences, sentences=random_sentences, gemini=True)

print("Accuracy:", result_with_gemini)

Running 5
Starting test number 1
Accuracy test 1 : 0.5
Starting test number 2
Accuracy test 2 : 0.6
Starting test number 3
Accuracy test 3 : 0.5
Starting test number 4
Accuracy test 4 : 0.7
Starting test number 5
Accuracy test 5 : 0.6
Accuracy: 0.58
